# ExitRadar API demo walkthrough

Start the API first: `PYTHONPATH=. uvicorn apps.api.app.main:app --port 8000`

In [ ]:
import httpx

BASE = "http://localhost:8000"
HEADERS = {"X-Demo-User": "00000000-0000-4000-8000-000000000001"}
WS = "00000000-0000-4000-8000-000000000010"

client = httpx.Client(base_url=BASE, headers=HEADERS, timeout=60.0)
print(client.get("/health").json())

In [ ]:
theses = client.get("/api/v1/theses", params={"workspace_id": WS}).json()
thesis = theses[0]
print(thesis["name"])
print(thesis["criteria"])

In [ ]:
cc = client.get(f"/api/v1/workspaces/{WS}/command-center", params={"thesis_id": thesis["id"]}).json()
print("opportunities", cc["opportunity_count"], "actionable", cc["newly_actionable"])
opps = client.get(f"/api/v1/theses/{thesis['id']}/opportunities").json()
for o in opps[:3]:
    print(o["opportunity_score"], o["company"]["canonical_name"], (o.get("recommendation") or {}).get("action"))

In [ ]:
top = opps[0]
detail = client.get(f"/api/v1/companies/{top['company_id']}/opportunities/{thesis['id']}").json()
print("Why now:", detail["explanation"]["why_now"])
print("Counter-signals:", detail["explanation"]["counter_signals"])
draft = client.post(f"/api/v1/companies/{top['company_id']}/outreach", json={"channel": "email", "thesis_id": thesis["id"]}).json()
print(draft["subject"])
print(draft["body"][:400])

In [ ]:
job = client.post("/api/v1/search-jobs", json={"thesis_id": thesis["id"], "location": "Phoenix, AZ", "max_results": 5}).json()
print(job["status"], job.get("stats"))